# 02 — Train and evaluate YOLOv8s on facade materials

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarEAbdelaal/M4U3-CV-Facade-Materials/blob/main/notebooks/02_training_eval.ipynb)

Detects 6 facade materials: `brick`, `cladding_panel`, `exposed_concrete`, `glass`, `painted_render`, `stone_cladding`.

**Reproducible with no credentials.** The dataset is downloaded from the project's GitHub Release and checked against its SHA256. No Roboflow account, API key or Colab Secret is needed.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

| Cell | What it does |
|---|---|
| 1 | Install the pinned Ultralytics version |
| 2 | Settings (model, epochs, image size, new-image URLs) |
| 3 | Download the frozen dataset from the GitHub Release and verify SHA256 |
| 3b | Optional: Roboflow download (skipped automatically when cell 3 worked) |
| 4 | Build this run's working copy of the dataset: polygons to boxes, drop boxes under 4 px, optionally merge per-pane boxes into per-area boxes |
| 5 | Record the environment (versions, GPU, time) |
| 6 | Train YOLOv8s |
| 7 | Evaluate: precision / recall / mAP50 / mAP50-95, overall and per class |
| 8 | Save training curves and confusion matrix |
| 9 | Predict on 10 validation images |
| 10 | Predict on new images that are not in the dataset |
| 11 | Package results and weights for download |
| 12 | Compare runs (when more than one run exists in this session) |

Outputs go to `/content/results/<RUN_NAME>/`, matching the repo's `results/` folder.

**Two runs, one comparison.** `MERGE_ELEMENTS` in cell 2 decides how the *training* labels are prepared:

- `True` (default, run name `yolov8s_merged`): boxes of `glass`, `stone_cladding` and `brick` that touch or nearly touch are merged into one box per continuous area, following the class-definition rule.
- `False` (run name `yolov8s_asis`): training labels as annotated (per pane / per stone).

Validation and test labels are **always** merged, so both runs are scored against the same answer key and their metrics can be compared. To run both: Run all with `True`, then set `False` in cell 2 and run from cell 2 to the end. Cell 12 then shows them side by side.

In [ ]:
# Cell 1 — install (version pinned for reproducibility)
%pip install -q ultralytics==8.4.163

In [ ]:
# Cell 2 — settings
from pathlib import Path

MODEL = "yolov8s.pt"      # pretrained on COCO; fine-tuned here (transfer learning)
EPOCHS = 50
IMGSZ = 640
BATCH = 16
SEED = 0
FRACTION = 1.0            # share of training images to use (1.0 = all)

# If no GPU is available: set QUICK_RUN = True for a 5-epoch verification run,
# and put the Release URL of the real best.pt in WEIGHTS_URL so evaluation and
# inference use the fully trained model (allowed by the brief; say so in the README).
QUICK_RUN = False
WEIGHTS_URL = ""
if QUICK_RUN:
    EPOCHS = 5

MIN_BOX_PX = 4            # boxes narrower or shorter than this (at 640 px) are annotation noise
CONF = 0.25               # confidence threshold for the prediction images

# 5 images that are NOT in the dataset (e.g. Wikimedia Commons file URLs), for the new-image evidence
NEW_IMAGE_URLS = [
    # "https://upload.wikimedia.org/...jpg",
]

# Label strategy (see docs/error_analysis.md, problem A3)
MERGE_ELEMENTS = True     # True: one box per continuous area for the classes below; False: labels as annotated
MERGE_CLASSES = ["glass", "stone_cladding", "brick"]
MERGE_GAP = 0.015         # same-class boxes closer than 1.5% of the image width count as one area
RUN_NAME = "yolov8s_merged" if MERGE_ELEMENTS else "yolov8s_asis"

RESULTS = Path("/content/results") / RUN_NAME
for sub in ["curves", "evidence/val_predictions", "evidence/new_predictions", "weights"]:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)
print(f"Run: {RUN_NAME} | {MODEL}, epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, quick_run={QUICK_RUN}")

## Dataset (primary, keyless path)

This is the course's standard download cell. Only `DATA_URL` and `SHA256` are project-specific.

- Release: `v1.0`, asset `facade-materials-v3-yolov8.zip` (contents: Roboflow project `facade-materials-3mrrt`, **version 4**)
- The checksum proves the file is the exact one the model was trained on.

In [ ]:
# Cell 3 — download the frozen dataset from the GitHub Release and verify it
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/OmarEAbdelaal/M4U3-CV-Facade-Materials/releases/download/v1.0/facade-materials-v3-yolov8.zip"
SHA256 = "134e41be8c310452bbe8ea33fd55a891c6e29ca863f3c9b8df3827f59bb951cd"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

## Dataset (optional secondary path: Roboflow)

Kept for the author's own workflow. On **Run all** it does nothing when the keyless dataset from cell 3 is already in place. It never crashes when no API key is set (it asks for one instead), and it is pinned to **version 4**, never "latest".

Roboflow's `yolov8` and `yolov11` exports use the same label and `data.yaml` format; `yolov8` matches the brief.

In [ ]:
# Cell 3b — secondary download from Roboflow (skipped when cell 3 succeeded)
from pathlib import Path

if not list(Path("/content/dataset").rglob("data.yaml")):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=True)
    try:
        from google.colab import userdata
        api_key = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        from getpass import getpass
        api_key = getpass("Roboflow API key (any account works for public Universe datasets): ")

    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)
    rf.workspace("omar-el-sayed-pzb69").project("facade-materials-3mrrt").version(4).download(
        "yolov8", location="/content/dataset_roboflow"
    )
    print("Downloaded to /content/dataset_roboflow. Point cfg_path at its data.yaml to use it.")
else:
    print("Keyless dataset already present; skipping Roboflow download.")

## Label preparation

Problems found when the released dataset was inspected (see `docs/project_log.md`, entry 8, and `docs/error_analysis.md`):

1. **Mixed label formats (A1).** Roboflow's YOLOv8 export kept SAM masks as polygons, and 181 label files mix polygons with plain boxes. The YOLOv8 detection loader then misreads the boxes. Fix: every polygon becomes its tightest box.
2. **Annotation fragments (A2).** About 3,500 boxes are under 4 px at 640 px. Fix: dropped.
3. **Inconsistent granularity (A3).** `glass`, `stone_cladding` and `brick` were labelled per pane / per stone (median 101, 76 and 38 boxes per image) while the other classes were labelled per area (3–10). Fix when `MERGE_ELEMENTS = True`: same-class boxes that touch or nearly touch are merged into one box per continuous area.

Each run gets a fresh working copy under `/content/work/<RUN_NAME>/`. The extracted release in `/content/dataset/` and the zip (and its checksum) are never changed.

In [ ]:
# Cell 4 — build this run's working copy of the dataset
import collections, shutil
import numpy as np, cv2, pandas as pd

names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]
merge_ids = {names.index(c) for c in MERGE_CLASSES}

def clean_boxes(lines, min_norm):
    """Polygons -> tightest box; drop malformed lines and boxes smaller than min_norm."""
    boxes, stats = [], collections.Counter()
    for line in lines:
        p = line.split()
        if not p:
            continue
        if len(p) > 5:
            xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
            x1, x2 = max(0.0, min(xs)), min(1.0, max(xs))
            y1, y2 = max(0.0, min(ys)), min(1.0, max(ys))
            p = [p[0], (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]
            stats["polygons_converted"] += 1
        if len(p) != 5:
            stats["malformed_dropped"] += 1
            continue
        c, cx, cy, w, h = int(p[0]), *map(float, p[1:])
        if min(w, h) < min_norm:
            stats["tiny_dropped"] += 1
            continue
        boxes.append([c, cx, cy, w, h])
    return boxes, stats

def merge_boxes(boxes, grid=640, gap=MERGE_GAP):
    """Merge same-class boxes that touch or nearly touch into one box per connected area.
    Each area's box is the union of the original boxes in it, so it never extends past them."""
    keep = [b for b in boxes if b[0] not in merge_ids]
    g = max(1, int(round(gap * grid)))
    for c in sorted(merge_ids):
        cls_boxes = [b for b in boxes if b[0] == c]
        if not cls_boxes:
            continue
        mask = np.zeros((grid, grid), np.uint8)
        corners = []
        for _, cx, cy, w, h in cls_boxes:
            x1, y1, x2, y2 = cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
            corners.append((x1, y1, x2, y2))
            px1, py1 = int(x1 * grid), int(y1 * grid)
            px2, py2 = int(np.ceil(x2 * grid)), int(np.ceil(y2 * grid))
            mask[max(0, py1 - g):min(grid, py2 + g), max(0, px1 - g):min(grid, px2 + g)] = 1
        _, labels = cv2.connectedComponents(mask, connectivity=8)
        areas = collections.defaultdict(list)
        for (x1, y1, x2, y2) in corners:
            px = min(grid - 1, max(0, int((x1 + x2) / 2 * grid)))
            py = min(grid - 1, max(0, int((y1 + y2) / 2 * grid)))
            areas[int(labels[py, px])].append((x1, y1, x2, y2))
        for members in areas.values():
            x1 = max(0.0, min(m[0] for m in members)); y1 = max(0.0, min(m[1] for m in members))
            x2 = min(1.0, max(m[2] for m in members)); y2 = min(1.0, max(m[3] for m in members))
            keep.append([c, (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1])
    return keep

WORK = Path("/content/work") / RUN_NAME
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(dataset_root, WORK, ignore=shutil.ignore_patterns("*.cache"))

stats, rows = collections.Counter(), []
for split in ["train", "valid", "test"]:
    lbl_dir = WORK / split / "labels"
    if not lbl_dir.is_dir():
        continue
    merge = MERGE_ELEMENTS if split == "train" else True   # val/test: always the common answer key
    before, after, imgs = collections.Counter(), collections.Counter(), collections.Counter()
    for lbl in lbl_dir.glob("*.txt"):
        boxes, st = clean_boxes(lbl.read_text().splitlines(), MIN_BOX_PX / IMGSZ)
        stats.update(st)
        for b in boxes:
            before[names[b[0]]] += 1
        if merge:
            boxes = merge_boxes(boxes)
        for b in boxes:
            after[names[b[0]]] += 1
        for c in {names[b[0]] for b in boxes}:
            imgs[c] += 1
        lbl.write_text("".join(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n" for b in boxes))
    for c in names:
        rows.append({"split": split, "class": c, "images": imgs[c],
                     "boxes_cleaned": before[c], "boxes_used": after[c]})
    print(f"{split}: {len(list(lbl_dir.glob('*.txt')))} images, merged={merge}")

work_cfg = dict(cfg, path=str(WORK))
work_cfg_path = WORK / "data.yaml"
work_cfg_path.write_text(yaml.safe_dump(work_cfg, sort_keys=False))
print(dict(stats))

counts = pd.DataFrame(rows).pivot(index="class", columns="split", values=["images", "boxes_cleaned", "boxes_used"])
counts.to_csv(RESULTS / "dataset_counts.csv")
counts

In [ ]:
# Cell 5 — record the environment (for the README "Reproducibility proof")
import json, platform, datetime, torch, ultralytics

run_info = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "python": platform.python_version(),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
    "quick_run": QUICK_RUN, "dataset_url": DATA_URL, "dataset_sha256": SHA256,
    "run_name": RUN_NAME, "merge_elements": MERGE_ELEMENTS, "merge_classes": MERGE_CLASSES,
    "merge_gap": MERGE_GAP, "min_box_px": MIN_BOX_PX,
}
(RESULTS / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(json.dumps(run_info, indent=2))
if not torch.cuda.is_available():
    print("\nNo GPU found. Runtime -> Change runtime type -> T4 GPU, or set QUICK_RUN = True in cell 2.")

In [ ]:
# Cell 6 — train
import time
from ultralytics import YOLO

t0 = time.time()
model = YOLO(MODEL)
model.train(
    data=str(work_cfg_path), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
    deterministic=True, fraction=FRACTION, plots=True,
    project="/content/runs", name=RUN_NAME, exist_ok=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
train_minutes = (time.time() - t0) / 60
print(f"Training took {train_minutes:.1f} min. Best weights: {BEST}")

if QUICK_RUN and WEIGHTS_URL:
    BEST = Path("/content/best_published.pt")
    urllib.request.urlretrieve(WEIGHTS_URL, BEST)
    print("QUICK_RUN: evaluation and inference use the published weights:", WEIGHTS_URL)

run_info["train_minutes"] = round(train_minutes, 1)
(RESULTS / "run_info.json").write_text(json.dumps(run_info, indent=2))

In [ ]:
# Cell 7 — evaluate: overall and per-class precision / recall / mAP
best = YOLO(str(BEST))
rows = []
for split in ["val"] + (["test"] if "test" in cfg else []):
    m = best.val(data=str(work_cfg_path), split=split, imgsz=IMGSZ, batch=BATCH, plots=True,
                 project="/content/runs", name=f"eval_{RUN_NAME}_{split}", exist_ok=True, verbose=False)
    rows.append({"run": RUN_NAME, "split": split, "class": "all", "precision": m.box.mp, "recall": m.box.mr,
                 "mAP50": m.box.map50, "mAP50-95": m.box.map})
    for i, c in enumerate(m.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        rows.append({"run": RUN_NAME, "split": split, "class": names[int(c)], "precision": p, "recall": r,
                     "mAP50": ap50, "mAP50-95": ap})
metrics = pd.DataFrame(rows).round(3)
metrics.to_csv(RESULTS / "metrics.csv", index=False)
metrics

In [ ]:
# Cell 8 — save training curves and confusion matrices
import shutil
from IPython.display import Image as IPImage, display

for f in list(RUN_DIR.glob("*.png")) + list(RUN_DIR.glob("*.csv")):
    shutil.copy(f, RESULTS / "curves" / f.name)
for split in ["val", "test"]:
    ev = Path("/content/runs") / f"eval_{RUN_NAME}_{split}"
    for f in ev.glob("confusion_matrix*.png"):
        shutil.copy(f, RESULTS / "curves" / f"{split}_{f.name}")
print(sorted(p.name for p in (RESULTS / "curves").iterdir()))

for name in ["results.png", "confusion_matrix_normalized.png"]:
    if (RESULTS / "curves" / name).exists():
        display(IPImage(filename=str(RESULTS / "curves" / name), width=900))

In [ ]:
# Cell 9 — predictions on 10 validation images (same 10 every run)
import random
from PIL import Image

val_images = sorted((WORK / "valid" / "images").glob("*"))
random.Random(SEED).shuffle(val_images)
for img in val_images[:10]:
    r = best.predict(str(img), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    Image.fromarray(r.plot()[:, :, ::-1]).save(RESULTS / "evidence" / "val_predictions" / f"{img.stem}_pred.jpg")
print(len(list((RESULTS / "evidence" / "val_predictions").glob("*.jpg"))), "validation predictions saved")

In [ ]:
# Cell 10 — predictions on new images that are not in the dataset
import requests

new_dir = Path("/content/new_images")
new_dir.mkdir(exist_ok=True)
headers = {"User-Agent": "M4U3-facade-materials/1.0 (student project)"}
for i, url in enumerate(NEW_IMAGE_URLS):
    dst = new_dir / f"new_{i:02d}.jpg"
    if not dst.exists():
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()
        dst.write_bytes(resp.content)

new_images = sorted(new_dir.glob("*.jpg"))
if not new_images:
    print("No new images: add 5 URLs to NEW_IMAGE_URLS in cell 2 (or upload files to /content/new_images).")
for img in new_images:
    r = best.predict(str(img), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    Image.fromarray(r.plot()[:, :, ::-1]).save(RESULTS / "evidence" / "new_predictions" / f"{img.stem}_pred.jpg")
print(len(list((RESULTS / "evidence" / "new_predictions").glob("*.jpg"))), "new-image predictions saved")

In [ ]:
# Cell 11 — package results and weights
shutil.copy(BEST, RESULTS / "weights" / "best.pt")
package = shutil.make_archive(f"/content/results_{RUN_NAME}", "zip", RESULTS)
print("results ->", package)
print("weights ->", RESULTS / "weights" / "best.pt", " (attach to a GitHub Release; do not commit it)")
try:
    from google.colab import files
    files.download(package)
except Exception:
    pass

In [ ]:
# Cell 12 — compare runs (each run's metrics.csv under /content/results/)
all_metrics = [pd.read_csv(f) for f in sorted(Path("/content/results").glob("*/metrics.csv"))]
if len(all_metrics) < 2:
    print("Only one run so far. Set MERGE_ELEMENTS = False in cell 2 and run from cell 2 to compare.")
comparison = pd.concat(all_metrics)
comparison = comparison[comparison["class"] == "all"].set_index(["split", "run"])[["precision", "recall", "mAP50", "mAP50-95"]]
comparison.to_csv("/content/results/comparison.csv")
comparison